In [1]:
import torch 
from transformers import GPT2LMHeadModel 
from model import GPT, GPTConfig, hf_key, _TRANSPOSED

sd_hf = GPT2LMHeadModel.from_pretrained('gpt2').state_dict()
model = GPT(GPTConfig())
sd_me = model.state_dict()

print(len(sd_me), len(sd_hf))

/Users/andresperezsoderi/nanogpt2/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 39708.12it/s]


161 149


In [7]:
counts = {
    'skipped': 0,
    'needs transpose': 0, 
    'straight copy': 0
}

for key in sd_me:
    if hf_key(key) is None:
        counts['skipped'] += 1
    elif hf_key(key).endswith(_TRANSPOSED):
        counts['needs transpose'] += 1
    else:
        counts['straight copy'] += 1

print(counts)


{'skipped': 12, 'needs transpose': 48, 'straight copy': 101}


In [8]:
new_count = 0 

for key in sd_me: 
    their = hf_key(key)
    if their is None:
        continue 
    elif their.endswith(_TRANSPOSED):
        source = sd_hf[their].t()
    else:
        source = sd_hf[their]
    assert sd_me[key].shape == source.shape
    new_count += 1
print(f"Number of keys processed: {new_count}")

Number of keys processed: 149


In [10]:
copied = 0 

with torch.no_grad():
    for key in sd_me: 
        their = hf_key(key)
        if their is None:
            continue 
        elif their.endswith(_TRANSPOSED):
            source = sd_hf[their].t()
        else:
            source = sd_hf[their]
        assert sd_me[key].shape == source.shape
        sd_me[key].copy_(source)
        copied+= 1
print(f"Number of keys copied: {copied}")

Number of keys copied: 149


In [ ]:
import tiktoken 
enc = tiktoken.get_encoding('gpt2')

text = "Hello, my dog is cute"
print(enc.n_vocab)
print(enc.encode(text))
print(enc.decode(enc.encode(text)))

50257
[15496, 11, 616, 3290, 318, 13779]
Hello, my dog is cute


In [13]:
model.eval()
prompt = "Hello, my dog is cute"
inputs = enc.encode(prompt)
inputs = torch.tensor(inputs, dtype=torch.long).view(1, -1)
out = model.generate(inputs, max_new_tokens=50)
print(enc.decode(out[0].tolist()))


Hello, my dog is cute.

Me and you can't go there."

Estebanze Facelleno had already been secret blogging for four years, before the social media spiral began.

"It will be boring. This is part of their
